***Project Overview***

SmartPantry AI is a food management application designed to help users track pantry items, reduce food waste, and improve food safety. The project combines recall monitoring, pantry management, and recipe recommendations into a single workflow. A key feature of SmartPantry is its ability to identify potentially recalled food products before they are used in meal recommendations.

The recall-monitoring component retrieves recall information from official sources and evaluates pantry items using product names, brands, and UPC codes. Results are classified as **CLEAR**, **POSSIBLE_MATCH**, or **RECALLED** to help users make informed decisions about the food they consume. As the project progresses, additional features and integrations will continue to improve accuracy, usability, and overall safety.

***Phase 1 – Research & Planning***

Defined the SmartPantry concept and project goals.
Researched pantry management and food recall systems.
Identified recall monitoring as a key project feature.

***Phase 2 – Design & Development Planning***

Designed the recall-monitoring workflow.
Defined recall classifications: CLEAR, POSSIBLE_MATCH, and RECALLED.
Planned integration with pantry and recipe modules.

In [1]:
from google.colab import userdata
import pandas as pd
import requests

# Read the key from Colab Secrets; the key itself is not shown.
try:
    api_key = userdata.get("OPENFDA_API_KEY")
except Exception as exc:
    raise RuntimeError(
        "Colab couldn't find OPENFDA_API_KEY. Check the Secrets panel "
        "and make sure Notebook access is enabled."
    ) from exc

if not api_key:
    raise RuntimeError(
        "The OPENFDA_API_KEY secret is empty. Add your key in the Secrets panel."
    )

# Ask openFDA for up to 5 food-recall records mentioning "milk".
url = "https://api.fda.gov/food/enforcement.json"
params = {
    "api_key": api_key,
    "search": "product_description:milk",
    "limit": 5,
}

response = requests.get(url, params=params, timeout=30)

if response.status_code == 200:
    results = response.json().get("results", [])
    recalls = pd.json_normalize(results)

    if recalls.empty:
        print("The request worked, but no matching records were returned.")
    else:
        columns_to_show = [
            "recall_number",
            "recalling_firm",
            "product_description",
            "reason_for_recall",
            "recall_initiation_date",
            "status",
        ]
        print(f"Success! Found {len(recalls)} matching record(s).")
        display(recalls.reindex(columns=columns_to_show).fillna(""))

elif response.status_code == 404:
    print("The request reached openFDA, but no records matched that search.")

elif response.status_code in (401, 403):
    print(
        "openFDA did not accept the request. Check that the API key was copied "
        "correctly into Colab Secrets and that the secret is named OPENFDA_API_KEY."
    )

else:
    print(f"The request returned HTTP status {response.status_code}.")
    print("Check your internet connection and try again.")


Success! Found 5 matching record(s).


,recall_number,recalling_firm,product_description,reason_for_recall,recall_initiation_date,status
0,F-1578-2018,Yukon Jackson,Coffee Toffee - 4 oz. packages. UPC 6 55974 89...,Coffee Toffee is recalled because pecan is lis...,20180525,Terminated
1,F-1233-2022,ELITE CONFECTIONERY LTD,ELITE MILK CHOCOLATE COATED RICE CAKES UPC:81...,Potential contamination with Salmonella,20220428,Terminated
2,F-2258-2019,Whole Flower Farms LLC,"Fudge Brownie Cookie, net wt. 2 oz. The produ...",Fudge Brownie Cookie product is recalled due t...,20190917,Terminated
3,F-2345-2014,Fish Family Farm,FISH FAMILY FARM INC. & & Shadow Valley Farm b...,Milk products and cream products may be contam...,20140612,Terminated
4,F-2254-2015,"Poco Dolce Confections, Inc.",Poco Dolce Bittersweet Chocolate: Ginger Tile...,California Department of Public Health conduct...,20150501,Terminated


In [2]:
pantry_item = "milk"

print(f"Checking recalls for: {pantry_item}")

Checking recalls for: milk


In [3]:
matches = recalls[
    recalls["product_description"]
    .str.contains(pantry_item, case=False, na=False)
]

print(matches[[
    "product_description",
    "reason_for_recall",
    "status"
]])

                                 product_description  \
0  Coffee Toffee - 4 oz. packages. UPC 6 55974 89...   
1  ELITE MILK CHOCOLATE COATED RICE CAKES  UPC:81...   
2  Fudge Brownie Cookie, net wt. 2 oz.  The produ...   
3  FISH FAMILY FARM INC. & & Shadow Valley Farm b...   
4  Poco Dolce Bittersweet Chocolate:  Ginger Tile...   

                                   reason_for_recall      status  
0  Coffee Toffee is recalled because pecan is lis...  Terminated  
1            Potential contamination with Salmonella  Terminated  
2  Fudge Brownie Cookie product is recalled due t...  Terminated  
3  Milk products and cream products may be contam...  Terminated  
4  California Department of Public Health conduct...  Terminated  


In [4]:
if len(matches) > 0:
    print("RECALLED")
else:
    print("CLEAR")

RECALLED


In [5]:
def check_recall(product_name, recalls_df):

    matches = recalls_df[
        recalls_df["product_description"]
        .str.contains(product_name, case=False, na=False)
    ]

    if len(matches) > 0:
        return "RECALLED"

    return "CLEAR"

In [6]:
print(check_recall("milk", recalls))

print(check_recall("banana", recalls))

RECALLED
CLEAR


In [7]:
import pandas as pd

def check_recall(product_name, recalls_df):

    matches = recalls_df[
        recalls_df["product_description"]
        .str.contains(product_name, case=False, na=False)
    ]

    if len(matches) > 0:
        return "RECALLED"

    return "CLEAR"


test_data = pd.DataFrame([
    {
        "product_description": "Milk Chocolate Rice Cakes"
    },
    {
        "product_description": "Frozen Pizza"
    }
])

print("Milk:", check_recall("milk", test_data))
print("Banana:", check_recall("banana", test_data))

Milk: RECALLED
Banana: CLEAR


In [8]:
import requests
import pandas as pd

def get_fda_recalls(api_key, search_term="milk", limit=5):
    url = "https://api.fda.gov/food/enforcement.json"
    params = {
        "api_key": api_key,
        "search": f"product_description:{search_term}",
        "limit": limit,
    }
    response = requests.get(url, params=params, timeout=30)
    if response.status_code == 200:
        results = response.json().get("results", [])
        return pd.json_normalize(results)
    return pd.DataFrame()

In [9]:
def check_recall(product_name, recalls_df):
    matches = recalls_df[
        recalls_df["product_description"]
        .str.contains(product_name, case=False, na=False)
    ]
    if len(matches) > 0:
        return "RECALLED"
    return "CLEAR"

In [10]:
import pandas as pd
import requests

def get_fda_recalls(api_key, search_term="milk", limit=5):

    url = "https://api.fda.gov/food/enforcement.json"

    params = {
        "api_key": api_key,
        "search": f"product_description:{search_term}",
        "limit": limit,
    }

    response = requests.get(url, params=params)

    if response.status_code == 200:
        results = response.json().get("results", [])
        return pd.json_normalize(results)

    return pd.DataFrame()


def check_recall(product_name, recalls_df):

    matches = recalls_df[
        recalls_df["product_description"]
        .str.contains(product_name, case=False, na=False)
    ]

    if len(matches) > 0:
        return "RECALLED"

    return "CLEAR"


# TEST
recalls = get_fda_recalls(
    api_key,
    search_term="milk",
    limit=20
)

print("Records Found:", len(recalls))

print("Milk:", check_recall("milk", recalls))
print("Banana:", check_recall("banana", recalls))

Records Found: 20
Milk: RECALLED
Banana: RECALLED


In [11]:
def check_recall(product_name, recalls_df):
    matches = recalls_df[
        recalls_df["product_description"]
        .str.contains(product_name, case=False, na=False)
    ]

    # Exact match
    for description in recalls_df["product_description"].fillna(""):
        if description.lower() == product_name.lower():
            return "RECALLED"

    # Partial match
    if len(matches) > 0:
        return "POSSIBLE_MATCH"

    return "CLEAR"

In [12]:
print("Milk:", check_recall("milk", recalls))
print("Chocolate:", check_recall("chocolate", recalls))
print("Banana:", check_recall("banana", recalls))
print(recalls["product_description"].head())


Milk: POSSIBLE_MATCH
Chocolate: POSSIBLE_MATCH
Banana: POSSIBLE_MATCH
0    Coffee Toffee - 4 oz. packages. UPC 6 55974 89...
1    ELITE MILK CHOCOLATE COATED RICE CAKES  UPC:81...
2    Fudge Brownie Cookie, net wt. 2 oz.  The produ...
3    FISH FAMILY FARM INC. & & Shadow Valley Farm b...
4    Poco Dolce Bittersweet Chocolate:  Ginger Tile...
Name: product_description, dtype: object


In [13]:
recalls = get_fda_recalls(
    api_key,
    search_term="product",
    limit=50
)

In [14]:
def check_recall(product_name, recalls_df):

    product_name = product_name.lower().strip()

    # Exact match
    for description in recalls_df["product_description"].fillna(""):

        description = description.lower()

        if description == product_name:
            return "RECALLED"

    # Partial match
    for description in recalls_df["product_description"].fillna(""):

        description = description.lower()

        if product_name in description:
            return "POSSIBLE_MATCH"

    return "CLEAR"

In [15]:
print("Milk:", check_recall("milk", recalls))
print("Chocolate:", check_recall("chocolate", recalls))
print("Banana:", check_recall("banana", recalls))
print("Laptop:", check_recall("laptop", recalls))

Milk: POSSIBLE_MATCH
Chocolate: POSSIBLE_MATCH
Banana: CLEAR
Laptop: CLEAR


In [16]:
def check_recall(product_name, recalls_df, brand=None):

    product_name = product_name.lower().strip()

    # Exact match
    for description in recalls_df["product_description"].fillna(""):

        description = description.lower()

        if description == product_name:
            return "RECALLED"

    # Brand + Product Match
    if brand:

        brand = brand.lower().strip()

        for description in recalls_df["product_description"].fillna(""):

            description = description.lower()

            if (
                brand in description
                and product_name in description
            ):
                return "POSSIBLE_MATCH"

    # Product-only partial match
    for description in recalls_df["product_description"].fillna(""):

        description = description.lower()

        if product_name in description:
            return "POSSIBLE_MATCH"

    return "CLEAR"

In [17]:
print(
    "Elite Chocolate:",
    check_recall(
        "chocolate",
        recalls,
        brand="elite"
    )
)

print(
    "Milk:",
    check_recall(
        "milk",
        recalls
    )
)

print(
    "Banana:",
    check_recall(
        "banana",
        recalls
    )
)

Elite Chocolate: POSSIBLE_MATCH
Milk: POSSIBLE_MATCH
Banana: CLEAR


In [19]:
def check_recall(product_name, recalls_df, brand=None):
    product_name = product_name.lower().strip()

    # Brand + Product Match
    if brand:
        brand = brand.lower().strip()
        for description in recalls_df["product_description"].fillna(""):
            description = description.lower()
            if brand in description and product_name in description:
                return "POSSIBLE_MATCH"

    # Product-only Match
    for description in recalls_df["product_description"].fillna(""):
        description = description.lower()
        if product_name in description:
            return "POSSIBLE_MATCH"

    return "CLEAR"

In [20]:
def check_recall(product_name, recalls_df, brand=None):
    # Brand + Product Match
    if brand:

        brand = brand.lower().strip()

        for description in recalls_df["product_description"].fillna(""):

            description = description.lower()

            if (
                brand in description
                and product_name in description
            ):
                return "POSSIBLE_MATCH"


    # Product-only Match
    for description in recalls_df["product_description"].fillna(""):

        description = description.lower()

        if product_name in description:
            return "POSSIBLE_MATCH"

    return "CLEAR"

In [21]:
print(
    "Elite Chocolate:",
    check_recall(
        "chocolate",
        recalls,
        brand="elite"
    )
)

print(
    "Milk:",
    check_recall(
        "milk",
        recalls
    )
)

print(
    "Banana:",
    check_recall(
        "banana",
        recalls
    )
)

print(
    "Laptop:",
    check_recall(
        "laptop",
        recalls
    )
)

Elite Chocolate: POSSIBLE_MATCH
Milk: POSSIBLE_MATCH
Banana: CLEAR
Laptop: CLEAR


In [22]:
{
    "status": "POSSIBLE_MATCH",
    "source": "FDA",
    "matched_product": "ELITE MILK CHOCOLATE COATED RICE CAKES"
}

{'status': 'POSSIBLE_MATCH',
 'source': 'FDA',
 'matched_product': 'ELITE MILK CHOCOLATE COATED RICE CAKES'}

In [23]:
"""
Returns a structured recall result.
"""

'\nReturns a structured recall result.\n'

In [24]:
def check_exact_match(description, product_name):
    if description == product_name:
        return {
            "status": "RECALLED",
            "source": "FDA",
            "matched_product": description
        }

In [25]:
def check_partial_match(description):
    return {
        "status": "POSSIBLE_MATCH",
        "source": "FDA",
        "matched_product": description
    }

In [26]:
def check_match_result(description):
    return {
        "status": "POSSIBLE_MATCH",
        "source": "FDA",
        "matched_product": description
    }

In [27]:
def get_clear_status():
    return {
        "status": "CLEAR",
        "source": None,
        "matched_product": None
    }

In [28]:
result = check_recall(
    "chocolate",
    recalls,
    brand="elite"
)

print(result)

POSSIBLE_MATCH


In [35]:
test_recalls = pd.DataFrame([
    {
        "product_description": "Milk Chocolate Rice Cakes",
        "upc": "123456789"
    },
    {
        "product_description": "Frozen Pizza",
        "upc": "987654321"
    }
])

In [37]:
def check_recall(product_name, recalls_df, brand=None, upc=None):
    # 1. UPC Match (Highest Confidence)
    if upc:
        for _, row in recalls_df.iterrows():
            recall_upc = str(row.get("upc", "")).strip()
            if recall_upc and recall_upc == str(upc):
                return {
                    "status": "RECALLED",
                    "source": "FDA",
                    "matched_product": row["product_description"]
                }

    # Normalize input text
    product_name_clean = product_name.lower().strip()

    # 2. Exact Match
    for _, row in recalls_df.iterrows():
        desc = str(row.get("product_description", "")).lower().strip()
        if desc == product_name_clean:
            return {
                "status": "RECALLED",
                "source": "FDA",
                "matched_product": row["product_description"]
            }

    # 3. Brand + Product Match
    if brand:
        brand_clean = brand.lower().strip()
        for _, row in recalls_df.iterrows():
            desc = str(row.get("product_description", "")).lower()
            if brand_clean in desc and product_name_clean in desc:
                return {
                    "status": "POSSIBLE_MATCH",
                    "source": "FDA",
                    "matched_product": row["product_description"]
                }

    # 4. Product-Only Match
    for _, row in recalls_df.iterrows():
        desc = str(row.get("product_description", "")).lower()
        if product_name_clean in desc:
            return {
                "status": "POSSIBLE_MATCH",
                "source": "FDA",
                "matched_product": row["product_description"]
            }

    # 5. CLEAR
    return {
        "status": "CLEAR",
        "source": None,
        "matched_product": None
    }

In [38]:
result = check_recall(
    "Chocolate Rice Cakes",
    test_recalls,
    brand="elite",
    upc="123456789"
)

print(result)

{'status': 'RECALLED', 'source': 'FDA', 'matched_product': 'Milk Chocolate Rice Cakes'}


In [39]:
result = check_recall(
    "Chocolate Rice Cakes",
    test_recalls,
    brand="elite",
    upc="555555555"
)

print(result)

{'status': 'POSSIBLE_MATCH', 'source': 'FDA', 'matched_product': 'Milk Chocolate Rice Cakes'}


In [41]:
import pandas as pd


def run_tests():

    recalls = pd.DataFrame([
        {
            "product_description": "Milk Chocolate Rice Cakes",
            "upc": "123456789"
        }
    ])

    print(
        "Test 1:",
        check_recall(
            "Chocolate Rice Cakes",
            recalls,
            upc="123456789"
        )
    )

    print(
        "Test 2:",
        check_recall(
            "chocolate",
            recalls,
            brand="elite"
        )
    )

    print(
        "Test 3:",
        check_recall(
            "banana",
            recalls
        )
    )


if __name__ == "__main__":
    run_tests()

Test 1: {'status': 'RECALLED', 'source': 'FDA', 'matched_product': 'Milk Chocolate Rice Cakes'}
Test 2: {'status': 'POSSIBLE_MATCH', 'source': 'FDA', 'matched_product': 'Milk Chocolate Rice Cakes'}
Test 3: {'status': 'CLEAR', 'source': None, 'matched_product': None}


In [43]:
import pandas as pd


def run_tests():

    recalls = pd.DataFrame([
        {
            "product_description": "Milk Chocolate Rice Cakes",
            "upc": "123456789"
        }
    ])

    # Test 1: UPC Match
    result = check_recall(
        "Chocolate Rice Cakes",
        recalls,
        upc="123456789"
    )

    assert result["status"] == "RECALLED"
    print("✅ Test 1 Passed: UPC Match")

    # Test 2: Brand + Product Match
    result = check_recall(
        "chocolate",
        recalls,
        brand="elite"
    )

    assert result["status"] == "POSSIBLE_MATCH"
    print("✅ Test 2 Passed: Brand Match")

    # Test 3: No Match
    result = check_recall(
        "banana",
        recalls
    )

    assert result["status"] == "CLEAR"
    print("✅ Test 3 Passed: Clear Result")


if __name__ == "__main__":
    run_tests()

✅ Test 1 Passed: UPC Match
✅ Test 2 Passed: Brand Match
✅ Test 3 Passed: Clear Result


In [44]:
    # Test 4: Unknown Product
    result = check_recall(
        "laptop",
        recalls
    )

    assert result["status"] == "CLEAR"
    print("✅ Test 4 Passed: Unknown Product")

    # Test 5: Product-Only Match
    result = check_recall(
        "milk",
        recalls
    )

    assert result["status"] == "POSSIBLE_MATCH"
    print("✅ Test 5 Passed: Product Match")

✅ Test 4 Passed: Unknown Product
✅ Test 5 Passed: Product Match


***Phase 3 – Recall Monitoring Implementation***

**Completed**

Connected to the FDA openFDA API.
Retrieved live recall records.
Implemented product, brand, and UPC matching.
Added structured recall result outputs.
Created and ran validation tests.

**Future Improvements**
Add USDA recall support.
Improve matching accuracy.
Complete integration with the SmartPantry application.

**Reflection**

The recall-monitoring system evolved from a project concept into a working prototype. The current implementation can retrieve live recall data, classify pantry items, and support SmartPantry's food-safety goals. Future development will focus on expanding data sources and supporting full application integration.